## Process Orders Data
1. Ingest the data into the data lakehouse - bronze_orders
2. Perform data quality checks and transform the data as required - silver_orders_clean
3. Explode the iterm array from the order object - silver_orders_clean_exploded
4. Select item attribute - silver_orders

![](./images/process-orders-data.png)

In [0]:
import dlt
import pyspark.sql.functions as F

#### 1. Ingest the data into the data lakehouse - bronze_orders



In [0]:
@dlt.table(
    name='bronze_orders',
    table_properties={'quality': 'bronze'},
    comment="Raw orders data ingested from the source system"
)
def create_bronze_orders():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format","json")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/circuitbox/landing/operational_data/orders/")
            .select(
                "*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.current_timestamp().alias("ingest_timestamp")
            )
    )

#### 2. Perform data quality checks and transform the data as required - silver_orders_clean


In [0]:
@dlt.table(
    name="silver_orders_clean",
    table_properties={'quality': 'silver'},
    comment="Cleaned orders data"
)
@dlt.expect_or_fail("valid_order_id", "order_id IS NOT NULL")
@dlt.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect("valid_order_status", "order_status IN ('Pending', 'Shipped', 'Cancelled', 'Completed')")
@dlt.expect("valid_payment_method", "payment_method IN ('Credit Card', 'Bank Transfer', 'PayPal')")
def create_silver_orders_clean():
    return (
        spark.readStream.table("bronze_orders")
            .select(
                "order_id",
                "customer_id",
                F.col("order_timestamp").cast("TIMESTAMP"),
                "payment_method",
                "items",
                "order_status"
            )
)

#### 3. Explode the iterm array from the order object - silver_orders_clean_exploded


In [0]:
@dlt.table(
    name="silver_orders_clean_exploded",
    table_properties={'quality': 'silver'},
    comment="Exploded items"
)
def create_silver_orders_clean_exploded():
    return (
        spark.readStream.table("silver_orders_clean")
            .select(
                "order_id",
                "customer_id",
                "order_timestamp",
                "payment_method",
                "order_status",
                F.explode("items").alias("item")
            )
    )



#### 4. Select item attribute - silver_orders

In [0]:
@dlt.table(
    name="silver_orders",
    table_properties={'quality': 'silver'},
    comment="Silver orders"
)
def create_silver_orders():
    return (
        spark.readStream.table("silver_orders_clean_exploded")
            .select(
                "order_id",
                "customer_id",
                "order_timestamp",
                "payment_method",
                "order_status",
                "item.item_id",
                F.col("item.name").alias("item_name"),
                F.col("item.price").alias("item_price"),
                F.col("item.quantity").alias("item_quantity"),
                F.col("item.category").alias("item_category")
            )
    )